# 04회 AICE Associate 모의고사 (종합, 실전처럼)
**시나리오: 인터넷은행 대출 연체 예측 (이진 분류)**

**이번엔 정답 보지 말고 60분 타이머 맞추고 끝까지 풀어 보세요.** 다 풀고 채점!
포인트: **결측 비율 높은 열 삭제, Y/N 타깃 변환, IQR 이상치, barplot, LogisticRegression, f1-score, 확률 예측**

| 컬럼 | 설명 |
|---|---|
| cust_no | 고객 번호 |
| age | 나이 |
| income | 연소득(만원, 이상치 있음) |
| job_type | 직업 (직장인/자영업/프리랜서/무직) |
| loan_amount | 대출금액(만원) |
| loan_term | 대출 기간(개월) |
| credit_score | 신용점수 |
| num_cards | 보유 카드 수 |
| app_visits | 월 앱 접속 횟수 |
| channel | 가입 경로 (앱/웹/제휴) |
| memo | 상담 메모 (대부분 비어 있음) |
| **default** | **연체 여부 (타깃, Y/N)** |

## 0. 준비 (문제 아님, 실행만 하세요)

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(31)
n = 4000
age = rng.integers(20, 66, n)
job = rng.choice(['직장인','자영업','프리랜서','무직'], n, p=[0.6,0.2,0.15,0.05])
income = np.round(rng.lognormal(8.2, 0.4, n) * np.select([job=='무직', job=='프리랜서'], [0.4, 0.8], 1), -1)
loan = np.round(rng.uniform(300, 5000, n), -1)
term = rng.choice([12, 24, 36, 60], n)
credit = np.clip(rng.normal(780, 90, n), 300, 1000).astype(int)
cards = rng.integers(0, 8, n)
visits = rng.poisson(6, n)
channel = rng.choice(['앱','웹','제휴'], n, p=[0.6,0.25,0.15])
logit = (-2.0 - 0.012*(credit-780) + 0.9*(loan/income) + 0.25*cards - 0.08*visits
         + 0.8*(job=='무직') + 0.4*(job=='프리랜서') + 0.3*(channel=='제휴'))
default = np.where(rng.random(n) < 1/(1+np.exp(-logit)), 'Y', 'N')
memo = np.where(rng.random(n) < 0.15, rng.choice(['재상담','서류보완','한도문의'], n), None)
df = pd.DataFrame({'cust_no': np.arange(1, n+1), 'age': age, 'income': income, 'job_type': job,
    'loan_amount': loan, 'loan_term': term, 'credit_score': credit, 'num_cards': cards,
    'app_visits': visits, 'channel': channel, 'memo': memo, 'default': default})
df.loc[rng.choice(n, 20, replace=False), 'income'] = 99999
df.loc[rng.choice(n, 80, replace=False), 'credit_score'] = np.nan
df.to_csv('loan_data.csv', index=False)
print('loan_data.csv 생성 완료')

## 1. 임포트와 데이터 불러오기

라이브러리 5개(`pd, np, sk, sns, plt`)를 임포트하고 `loan_data.csv`를 **`df`** 로 읽으세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
import pandas as pd
import numpy as np
import sklearn as sk
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv('loan_data.csv')
df.head()
```


</details>

## 2. 결측치 비율 확인

각 컬럼의 **결측치 비율(%)** 을 계산해 출력하세요. 비율이 **50%가 넘는 컬럼**은?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
(df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
# 또는 df.isnull().mean() * 100
```

답: **memo** (약 85%).

`isnull().mean()`은 True를 1로 보고 평균을 내니까 곧바로 비율이 나와요.
</details>

## 3. 컬럼 삭제

`cust_no`와 결측이 너무 많은 `memo`를 삭제해 **`df1`** 에 저장하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df1 = df.drop(columns=['cust_no', 'memo'])
```

결측이 절반 넘는 열은 채우기보다 삭제하는 게 보통이에요.
</details>

## 4. 타깃 변환

`default`의 **Y는 1, N은 0** 으로 바꾸세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df1 = df1.copy()
df1['default'] = df1['default'].map({'Y': 1, 'N': 0})
df1['default'].value_counts()
```

답: 0이 2,417개, 1이 1,583개. 모델은 Y/N 글자를 못 읽어서 숫자로 바꿔야 해요. `replace({'Y':1,'N':0})`도 정답.
</details>

## 5. 시각화 (1) 범주별 비율

seaborn **barplot** 으로 `job_type`(x)별 `default`(y)의 평균(=연체율)을 그리세요. 연체율이 **가장 높은 직업**은?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
sns.barplot(data=df1, x='job_type', y='default')
plt.show()
df1.groupby('job_type')['default'].mean().sort_values()
```

답: **무직**.

barplot은 y의 **평균**을 막대로 그려요. 0/1 타깃의 평균 = 비율이라서 연체율이 바로 보여요.
</details>

## 6. 시각화 (2) 숫자 분포 비교

seaborn **histplot** 으로 `credit_score` 분포를 `hue='default'`로 나눠 그리세요. 연체(1) 고객의 신용점수는 **높은 쪽, 낮은 쪽** 중 어디에 몰려 있나요?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
sns.histplot(data=df1, x='credit_score', hue='default', kde=True)
plt.show()
```

답: **낮은 쪽**. `hue`는 색깔로 그룹을 나눠 주는 옵션이에요. 시각화 문제에서 자주 써요.
</details>

## 7. 결측치 처리

`credit_score` 결측치를 **중앙값**으로 채우세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
df1['credit_score'] = df1['credit_score'].fillna(df1['credit_score'].median())
df1.isnull().sum().sum()
```

답: 결측치 합계 **0**.
</details>

## 8. 이상치 제거 (IQR)

`income`에 대해 **IQR 방식**으로 이상치를 제거해 **`df2`** 에 저장하세요.
- Q1 = 25%, Q3 = 75%, IQR = Q3 - Q1
- 정상 범위: Q1 - 1.5×IQR 이상, Q3 + 1.5×IQR 이하

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
Q1 = df1['income'].quantile(0.25)
Q3 = df1['income'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df2 = df1[(df1['income'] >= lower) & (df1['income'] <= upper)]
print(len(df1), '→', len(df2))
```

IQR 공식은 시험에 그대로 나올 수 있으니 통째로 외우세요. 조건 두 개는 **각각 괄호** 치고 `&`로 연결!

(99999 외에 원래 분포 끝에 있던 고소득자도 몇 명 같이 빠져요. 정상이에요.)
</details>

## 9. 원핫 인코딩과 분리

`job_type`, `channel`을 원핫 인코딩(`drop_first=True`) → **`df_preset`**. `default`를 y로 **test 25%**, stratify, `random_state=42`로 분리하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.model_selection import train_test_split

df_preset = pd.get_dummies(df2, columns=['job_type', 'channel'], drop_first=True)
X = df_preset.drop('default', axis=1)
y = df_preset['default']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
```

test_size가 0.2가 아니라 **0.25** 인 것처럼, 숫자 조건을 꼼꼼히 읽는 게 실전 감점 포인트예요.
</details>

## 10. 스케일링

StandardScaler로 스케일링하세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
```


</details>

## 11. 머신러닝 모델 3개

다음 모델을 학습하세요.
- **LogisticRegression** (`max_iter=1000`) → `lg`
- **DecisionTreeClassifier** (`max_depth=4, random_state=42`) → `dt`
- **RandomForestClassifier** (`n_estimators=200, max_depth=8, random_state=42`) → `rf`

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

lg = LogisticRegression(max_iter=1000).fit(X_train, y_train)
dt = DecisionTreeClassifier(max_depth=4, random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42).fit(X_train, y_train)
```

LogisticRegression은 이름에 Regression이 붙어 있지만 **분류 모델**이에요. 단골 함정!

`.fit()`을 바로 이어 붙여도 같은 결과예요.
</details>

## 12. 평가 (f1-score)

세 모델의 **accuracy와 f1-score** 를 비교하세요. **f1 기준 최고 모델**은?

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from sklearn.metrics import accuracy_score, f1_score

for name, m in [('LG', lg), ('DT', dt), ('RF', rf)]:
    pred = m.predict(X_test)
    print(name, round(accuracy_score(y_test, pred), 3), round(f1_score(y_test, pred), 3))
```

답: **RandomForest** (f1 약 0.65, LogisticRegression 0.64로 근소한 2등, DecisionTree 0.56). 1이 적은 데이터는 accuracy만 보면 속아요. 다 0이라고 찍어도 60%가 나오니까요. 그래서 **f1** 을 같이 봐요.
</details>

## 13. 딥러닝 이진 분류

DNN을 만들고 학습하세요.
- Dense(32, relu) → Dropout(0.2) → Dense(16, relu) → **이진 분류 출력층**
- adam, 이진 분류 loss, metrics `['accuracy']`
- EarlyStopping(`monitor='val_loss', patience=5, restore_best_weights=True`)
- epochs=50, batch_size=64, `validation_split=0.2`, 결과 `history`

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=50, batch_size=64,
                    validation_split=0.2, callbacks=[es])
```

`restore_best_weights=True`는 멈출 때 가장 좋았던 순간으로 되돌려 주는 옵션이에요.
</details>

## 14. 평가와 학습 곡선

1) DNN으로 X_test를 예측해 **0.5 기준으로 0/1** 로 바꾸고 f1-score를 출력하세요.
2) `accuracy`와 `val_accuracy` 그래프를 그리세요.

In [ ]:
# 여기에 직접 작성해 보세요

<details><summary><b>👉 정답 보기</b></summary>

```python
proba = model.predict(X_test)
pred = (proba > 0.5).astype(int)      # 확률 → 0/1
print('DNN f1:', f1_score(y_test, pred))

plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.xlabel('Epochs'); plt.ylabel('Accuracy')
plt.legend(); plt.show()
```

sigmoid 출력은 0~1 확률이에요. **0.5보다 크면 1** 로 바꿔야 f1 같은 지표를 계산할 수 있어요.

수고했어요! 몇 문제 맞혔는지 세어 보고, 틀린 문제만 01~03회에서 비슷한 걸 다시 풀어 보세요.
</details>

---
# 🏁 마무리 체크리스트
- 막히면 `함수이름?` 실행 또는 괄호 안에서 **Shift + Tab**
- `train_test_split` 반환 순서: **X_train, X_test, y_train, y_test**
- test 데이터는 **transform만** (fit 금지)
- 출력층과 loss 짝꿍 확인 (이진 sigmoid+binary_crossentropy / 다중 softmax+sparse_categorical_crossentropy / 회귀 Dense(1)+mse)
- 문제에서 정한 **변수명** 그대로 쓰기
- 다 풀었으면 정답 안 보고 **60분 타이머** 맞춰 한 번 더!